In [ ]:
import os
import pandas as pd
import numpy as np
from pyexpat import features
import neural_ode
from neural_ode import generate_synthetic_data, prep_data, prep_batches, create_model
from neural_ode import train_pretrain, train_fit_gmm, train_finetune, predict, evaluate_performance
from neural_ode import plot_input_trajectories, plot_predicted_trajectories, plot_latent_space, plot_training_loss


### Create dataset

In [ ]:
train_data_df, test_data_df = generate_synthetic_data(
    n_train=300, n_test=100, n_classes=3, n_features=4,
    length=60, irregularity=0.5, noise=0.2, seed=42,
)


### Examine dataset

In [ ]:
print(f"Rows: {len(train_data_df)}")
print(f"Trajectories: {train_data_df['trajectory_id'].nunique()}")
print(f"Labels: {train_data_df['label'].nunique()}")
print(f"Features: {[c for c in train_data_df.columns if c.startswith('feature_')]}")
print(f"Rows per trajectory:\n{train_data_df.groupby('trajectory_id').size().describe().round(1)}")

### Prep Dataset

In [ ]:
[c for c in train_data_df.columns if c.startswith('feature_')]

In [ ]:
data = prep_data(
    train_data_df, id_col="trajectory_id", 
    time_col="time",
    feature_cols= [c for c in train_data_df.columns if c.startswith('feature_')], 
    label_col="label",
    cutoff=3.5, 
    cutoff_mode="time",
)

plot_input_trajectories(data)

loader = prep_batches(data, batch_size=16, val_split=0.2, normalize=True)


In [ ]:
model = create_model(len(data.feature_names), {
    "model_name": "LatentODE_odernn",
    "latent_dim": 8,
    "k": 3,
    "method": "euler",
    "networks": {
        "z0_encoder_odernn": {
            "state_dim": 64,
            "recognition_ode_hidden_dims": [32],
            "gru_update_hidden_dims": [64],
            "posterior_hidden_dims": [32],
        },
        "generative_ode": {"hidden_dims": [32]},
        "emission": {"hidden_dims": [16]},
    },
})



In [ ]:
pretrain_result = train_pretrain(loader, model_config=model, epochs=200)

plot_training_loss(pretrain_result)
plot_latent_space(pretrain_result.model, loader, color_by="ground_truth")

predictions_pre = predict(pretrain_result.model, loader)
plot_predicted_trajectories(predictions_pre, max_trajectories_per_group=15, show_observed = True)

In [ ]:
gmm_result = train_fit_gmm(pretrain_result.model, loader)

plot_latent_space(gmm_result.model, loader, color_by = "ground_truth")
plot_latent_space(gmm_result.model, loader, color_by = "cluster")

In [ ]:
finetune_result = train_finetune(gmm_result.model, loader, epochs=50)

plot_training_loss(finetune_result)
plot_latent_space(finetune_result.model, loader, color_by="cluster")

predictions = predict(finetune_result.model, loader)
plot_predicted_trajectories(predictions, max_trajectories_per_group=15, show_observed = True)

### Prep test data

In [ ]:
test_data = prep_data(
    test_data_df, id_col="trajectory_id", 
    time_col="time",
    feature_cols=[c for c in train_data_df.columns if c.startswith('feature_')], 
    label_col="label",
    cutoff=3.5, 
    cutoff_mode="time"
    )
plot_input_trajectories(test_data)
test_loader = prep_batches(test_data, batch_size=16, val_split=0, normalize=True)

plot_latent_space(finetune_result.model, test_loader, color_by="cluster")


In [ ]:

predictions = predict(finetune_result.model, loader)
plot_predicted_trajectories(predictions, max_trajectories_per_group=150, show_observed = True)

scores = evaluate_performance(finetune_result.model, test_loader.train)
scores